# Fase 3 — Análisis estadístico

Lee `results/metrics.csv` y ejecuta:

| Prueba | Datos | Comparación |
|--------|-------|-------------|
| Wilcoxon pareado | `same_seed` | original vs modificado, por semilla |
| Mann-Whitney U | `diff_seed` | original vs modificado, independiente |

Para cada N ∈ {50, 100, 150, 200} y cada métrica (HV, GD, IGD).  
Corrección de Holm-Bonferroni sobre los p-valores de cada métrica.  
Tamaño del efecto: Vargha-Delaney Â₁₂ (via `pingouin`).

In [ ]:
import pandas as pd
import numpy as np
import pingouin as pg
from statsmodels.stats.multitest import multipletests

RESULTS_BASE = '../results'
N_VALUES     = [50, 100, 150, 200]
METRICS      = ['HV', 'GD', 'IGD']

In [ ]:
df = pd.read_csv(f'{RESULTS_BASE}/metrics.csv')
print(f'Corridas cargadas: {len(df)}')
df.groupby(['variante', 'seed_mode', 'N']).size().rename('n').to_frame()

## 1. Pruebas estadísticas por métrica

In [ ]:
def run_tests(df, metric):
    rows = []
    for n in N_VALUES:
        orig_same = df[(df.variante == 'original')  & (df.seed_mode == 'same_seed') & (df.N == n)]
        mod_same  = df[(df.variante == 'modified')  & (df.seed_mode == 'same_seed') & (df.N == n)]
        orig_diff = df[(df.variante == 'original')  & (df.seed_mode == 'diff_seed') & (df.N == n)]
        mod_diff  = df[(df.variante == 'modified')  & (df.seed_mode == 'diff_seed') & (df.N == n)]

        # ── Wilcoxon pareado (same_seed) ──────────────────────────────────
        if len(orig_same) > 0 and len(mod_same) > 0:
            merged = orig_same.merge(mod_same, on='seed', suffixes=('_orig', '_mod'))
            if len(merged) >= 2:
                res_w = pg.wilcoxon(merged[f'{metric}_orig'], merged[f'{metric}_mod'])
                p_w   = float(res_w['p-val'].iloc[0])
                a12_w = float(res_w['CLES'].iloc[0]) if 'CLES' in res_w.columns else np.nan
            else:
                p_w, a12_w = np.nan, np.nan
        else:
            p_w, a12_w = np.nan, np.nan

        rows.append({'N': n, 'test': 'Wilcoxon', 'seed_mode': 'same_seed',
                     'p_valor': p_w, 'A12': a12_w})

        # ── Mann-Whitney U (diff_seed) ────────────────────────────────────
        if len(orig_diff) >= 2 and len(mod_diff) >= 2:
            res_mw = pg.mwu(orig_diff[metric], mod_diff[metric], alternative='two-sided')
            p_mw   = float(res_mw['p-val'].iloc[0])
            a12_mw = float(res_mw['CLES'].iloc[0]) if 'CLES' in res_mw.columns else np.nan
        else:
            p_mw, a12_mw = np.nan, np.nan

        rows.append({'N': n, 'test': 'Mann-Whitney', 'seed_mode': 'diff_seed',
                     'p_valor': p_mw, 'A12': a12_mw})

    result_df = pd.DataFrame(rows)

    # ── Holm-Bonferroni sobre todos los p-valores de esta métrica ─────────
    valid_mask = result_df['p_valor'].notna()
    if valid_mask.sum() > 0:
        pvals = result_df.loc[valid_mask, 'p_valor'].values
        reject, pvals_corr, _, _ = multipletests(pvals, method='holm')
        result_df.loc[valid_mask, 'p_corr_holm'] = pvals_corr
        result_df.loc[valid_mask, 'reject_H0']   = reject
    else:
        result_df['p_corr_holm'] = np.nan
        result_df['reject_H0']   = np.nan

    result_df.insert(0, 'metrica', metric)
    return result_df

all_results = pd.concat([run_tests(df, m) for m in METRICS], ignore_index=True)
all_results

## 2. Guardar resultados

In [ ]:
out_path = f'{RESULTS_BASE}/statistical_results.csv'
all_results.to_csv(out_path, index=False)
print(f'Guardado: {out_path}')

## 3. Tabla de resultados por métrica

In [ ]:
def fmt_pval(p):
    if pd.isna(p): return 'N/A'
    if p < 0.001: return '< 0.001'
    return f'{p:.4f}'

for metric in METRICS:
    sub = all_results[all_results.metrica == metric].copy()
    sub['p_valor_fmt']    = sub['p_valor'].map(fmt_pval)
    sub['p_corr_holm_fmt'] = sub['p_corr_holm'].map(fmt_pval)
    sub['A12_fmt']        = sub['A12'].map(lambda x: f'{x:.4f}' if pd.notna(x) else 'N/A')
    sub['significativo']  = sub['reject_H0'].map(lambda x: '✓' if x == True else ('✗' if x == False else 'N/A'))
    print(f'\n{'='*60}')
    print(f'Métrica: {metric}')
    print('='*60)
    display_cols = ['N', 'test', 'seed_mode', 'p_valor_fmt', 'p_corr_holm_fmt', 'A12_fmt', 'significativo']
    print(sub[display_cols].to_string(index=False))

## 4. Interpretación de Â₁₂

| Â₁₂ | Interpretación |
|------|---------------|
| ≈ 0.5 | Sin efecto (equivalente) |
| 0.56 – 0.64 | Efecto pequeño |
| 0.64 – 0.71 | Efecto mediano |
| > 0.71 | Efecto grande |

Â₁₂ > 0.5 indica que el **modificado** supera al original con esa probabilidad (si comparamos original primero).

In [ ]:
pivot = all_results.pivot_table(
    index=['metrica', 'N'],
    columns='test',
    values=['p_valor', 'p_corr_holm', 'A12', 'reject_H0']
)
pivot